# semantic_search/03 — What do the clusters correspond to clinically?

Tests every available clinical characteristic against each partition from `02_cluster`, describes
each cluster's survival, and plots the result.

**Runs after** `semantic_search/02_cluster`. Nothing depends on it.

## What gets tested

| Family | Variables | Test |
|---|---|---|
| `demographics` | age, gender | Kruskal-Wallis / chi-square |
| `cancer_type` | `CANCER_TYPE` | chi-square |
| `stage` | major stage I-IV | chi-square |
| `met_burden` | `N_MET_SITES`, 8 site indicators | Kruskal / chi-square |
| `treatment` | `PX_on_<class>` | chi-square |
| `somatic` | `<GENE>_{SNV,AMP,DEL,SV,FUSION}` | Fisher when sparse, else chi-square |
| `prs` | `PGS*` | Kruskal |
| `survival` | `tt_death`, `death` | logrank + Cox |
| `note_volume` | notes per type, span, mean year | Kruskal |

**BH-FDR is applied within each family, not pooled**, mirroring `_fdr_within_mutation_type` in
`run_IPTW_analysis.py`. The somatic and PRS families carry hundreds of tests each; pooling them with
the two demographic tests would bury the latter under a correction they did not earn.

**This repo has no labs, vitals, race/ethnicity, smoking, ECOG, or PFS.** Age and gender are the
only demographics, and OS is the only time-to-event endpoint. That is the full extent of
"all available clinical characteristics" here.

## The two results that matter most

**`cluster_note_volume.csv` is a confound check, not a finding.** An unweighted mean over a
patient's notes encodes *how much* was written about them as well as *what* was written. If clusters
separate mainly on note counts, the partition is a documentation-intensity artifact.

**The adjusted Cox is the load-bearing survival number.** Pathology notes name the tumor, so a
cluster/OS association that vanishes after adjusting for cancer type is a restatement of the
diagnosis rather than a new axis. `cox_hr` vs `cox_hr_adjusted` is where you see that happen.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<24} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a semantic_search stage as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={time.time() - t0:,.1f}s", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

Prevalence floors for the wide families (`MIN_SOMATIC_PREVALENCE`, `MIN_TREATMENT_PREVALENCE`) and
the join-coverage warning threshold are module constants in `clinical_data.py` and
`characterize_clusters.py` — change them there.

In [ ]:
MODULE = "semantic_search.characterize_clusters"

SPACES = common.SPACES
WINDOWS = common.WINDOWS

# Which partition the figures below illustrate. The CSVs cover every space.
FOCUS_SPACE = "merged"
FOCUS_WINDOW = "pretreatment"

print(f"testing: {len(SPACES)} spaces x {len(WINDOWS)} windows")
print(f"figures: {FOCUS_SPACE}/{FOCUS_WINDOW}")

## Preconditions

Cluster labels plus the covariate files. Missing covariate files degrade to a skipped family, not a crash.

In [ ]:
missing = check_inputs(
    [(f"labels {s}/{w}", common.labels_path(s, w)) for w in WINDOWS for s in SPACES]
    + [
        ("survival cohort", os.path.join(config.SURV_PATH, "death_met_surv_df.parquet")),
        ("cancer types",    os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz")),
        ("cancer stage",    os.path.join(config.FEATURE_PATH, "cancer_stage_df.csv.gz")),
        ("met burden",      os.path.join(config.FEATURE_PATH, "met_burden_df.csv.gz")),
        ("treatment lines", os.path.join(config.FEATURE_PATH,
                                         "categorical_treatment_data_by_line.csv.gz")),
        ("somatic",         os.path.join(config.FEATURE_PATH, "complete_somatic_data_df.csv.gz")),
        ("germline PRS",    os.path.join(config.FEATURE_PATH, "complete_germline_data_df.csv.gz")),
    ]
)

## Run

In [ ]:
rc = run_module(MODULE, ["--spaces", *SPACES, "--windows", *WINDOWS])
if rc != 0:
    print("\nStage failed - see the traceback above.")

## Join coverage

Read this before any result below. A family that matched only a fraction of clustered patients is
describing a subset, and its p-values are about that subset.

In [ ]:
import polars as pl

cov_path = common.result_path("cluster_join_coverage")
if os.path.exists(cov_path):
    cov = pl.read_csv(cov_path).filter(
        (pl.col("space") == FOCUS_SPACE) & (pl.col("window") == FOCUS_WINDOW))
    for r in cov.sort("coverage").iter_rows(named=True):
        flag = "  <-- LOW" if r["below_threshold"] else ""
        print(f"  {r['family']:14s} {r['n_matched']:>7,d}/{r['n_clustered']:<7,d} "
              f"= {r['coverage']:6.1%}{flag}")

## What separates the clusters

FDR-significant variables, most significant first, for every space.

In [ ]:
res_path = common.result_path("cluster_vs_clinical")
if os.path.exists(res_path):
    res = pl.read_csv(res_path)
    hits = res.filter(pl.col("significant") == True)  # noqa: E712
    print(f"{hits.height} of {res.height} tests significant at FDR < 0.05\n")
    with pl.Config(tbl_rows=40, tbl_width_chars=170):
        print(hits.sort("fdr").select(
            ["space", "window", "family", "variable", "test", "p", "fdr"]))

## Confound check: is this just documentation volume?

Any significant `note_volume` row means the partition is partly tracking how much was written about
each patient rather than what it said. That does not invalidate the clusters, but it has to be
reported alongside them.

In [ ]:
nv_path = common.result_path("cluster_note_volume")
if os.path.exists(nv_path):
    nv = pl.read_csv(nv_path)
    sig = nv.filter(pl.col("significant") == True)  # noqa: E712
    if sig.height:
        print(f"WARNING: {sig.height} note-volume variables separate the clusters:\n")
        with pl.Config(tbl_rows=30, tbl_width_chars=150):
            print(sig.sort("fdr").select(["space", "window", "variable", "p", "fdr"]))
        print("\nClusters are partly a documentation-intensity artifact. Consider")
        print("residualizing note count out, or switching to time_decay_mean pooling.")
    else:
        print("No note-volume variable separates the clusters at FDR < 0.05.")
        print("Documentation intensity does not explain the partitions.")

## Survival by cluster

`cox_hr` is crude; `cox_hr_adjusted` adds age, gender and cancer type. The reference is the largest
cluster in both, so the two are directly comparable. **A large gap between them means cancer type
explains the survival difference.**

In [ ]:
surv_path = common.result_path("cluster_survival")
if os.path.exists(surv_path):
    surv = pl.read_csv(surv_path)
    with pl.Config(tbl_rows=40, tbl_width_chars=190):
        print(surv.select(["space", "window", "cluster", "n", "n_events",
                           "median_os_months", "rmst_months", "logrank_p",
                           "cox_hr", "cox_p", "cox_hr_adjusted", "cox_p_adjusted"]))

## Figures

Exploratory, drawn inline with matplotlib — this arm has no R tier. Panels are also saved under
`SEMANTIC_SEARCH_PATH/figures/`, and a manifest records every panel including the ones skipped
and why.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from figures.io import apply_style
from shared.palette import CLUSTER_COLORS

apply_style()
common.ensure_dirs()

manifest = []


def record(name, status, reason=""):
    manifest.append({"panel": name, "status": status, "reason": reason})
    if status != "ok":
        print(f"  [skip] {name}: {reason}")


def cluster_color(i):
    return CLUSTER_COLORS[i % len(CLUSTER_COLORS)]


def save(fig, name):
    path = common.figure_path(name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    record(name, "ok")
    return path

### Cluster structure vs cancer type

The single most informative panel in this notebook. Left is the partition; right is the same
scatter colored by diagnosis. **If the two look alike, the clusters are restating cancer type** —
which the adjusted Cox above should independently confirm.

In [ ]:
from semantic_search import clinical_data

name = "scatter_cluster_vs_cancer_type"
if not os.path.exists(common.coords_path(FOCUS_SPACE, FOCUS_WINDOW)):
    record(name, "skipped", "no coords for the focus space")
else:
    coords = common.load_coords(FOCUS_SPACE, FOCUS_WINDOW)
    labels = common.load_labels(FOCUS_SPACE, FOCUS_WINDOW)
    ct_df, _, _ = clinical_data.load_cancer_type()
    plot_df = coords.join(labels, on=common.PATIENT_KEY).join(
        ct_df, on=common.PATIENT_KEY, how="left")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    x = plot_df.get_column("dim1").to_numpy()
    y = plot_df.get_column("dim2").to_numpy()

    clusters = sorted(plot_df.get_column("cluster").unique().to_list())
    for i, cl in enumerate(clusters):
        m = plot_df.get_column("cluster").to_numpy() == cl
        axes[0].scatter(x[m], y[m], s=5, alpha=0.5, color=cluster_color(i),
                        label=f"cluster {cl} (n={int(m.sum()):,})", rasterized=True)
    axes[0].set_title(f"Clusters - {FOCUS_SPACE} / {FOCUS_WINDOW}")
    axes[0].legend(fontsize=7, markerscale=2, loc="best")

    if "CANCER_TYPE" in plot_df.columns:
        types = plot_df.get_column("CANCER_TYPE").fill_null("UNKNOWN")
        top = [t for t, _ in sorted(
            types.value_counts().rows(), key=lambda r: -r[1])[:10]]
        cmap = plt.get_cmap("tab10")
        arr = types.to_numpy()
        other = ~np.isin(arr, top)
        axes[1].scatter(x[other], y[other], s=4, alpha=0.25, color="0.75",
                        label="other", rasterized=True)
        for i, t in enumerate(top):
            m = arr == t
            axes[1].scatter(x[m], y[m], s=5, alpha=0.6, color=cmap(i % 10),
                            label=f"{t} (n={int(m.sum()):,})", rasterized=True)
        axes[1].set_title("Same points, colored by cancer type")
        axes[1].legend(fontsize=6, markerscale=2, loc="best")

    for ax in axes:
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
    fig.tight_layout()
    save(fig, name)
    plt.show()

### Cluster composition by cancer type and stage

In [ ]:
for var, loader, panel in [
    ("CANCER_TYPE", clinical_data.load_cancer_type, "composition_cancer_type"),
    ("CANCER_STAGE", clinical_data.load_stage, "composition_stage"),
]:
    df, _, _ = loader()
    if not df.height or var not in df.columns:
        record(panel, "skipped", f"{var} unavailable")
        continue
    labels = common.load_labels(FOCUS_SPACE, FOCUS_WINDOW)
    merged = labels.join(df, on=common.PATIENT_KEY, how="left").drop_nulls(var)
    if not merged.height:
        record(panel, "skipped", f"no {var} matched the clustered patients")
        continue

    counts = merged.group_by(["cluster", var]).len(name="n")
    levels = [r[0] for r in sorted(
        merged.get_column(var).value_counts().rows(), key=lambda r: -r[1])[:12]]
    clusters = sorted(merged.get_column("cluster").unique().to_list())

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bottom = np.zeros(len(clusters))
    cmap = plt.get_cmap("tab20")
    for i, lv in enumerate(levels):
        vals = []
        for cl in clusters:
            row = counts.filter((pl.col("cluster") == cl) & (pl.col(var) == lv))
            total = counts.filter(pl.col("cluster") == cl).get_column("n").sum()
            vals.append(100.0 * (row.get_column("n").sum() if row.height else 0) / total)
        vals = np.array(vals)
        ax.bar([str(c) for c in clusters], vals, bottom=bottom,
               color=cmap(i % 20), label=str(lv))
        bottom += vals
    ax.set_xlabel("cluster")
    ax.set_ylabel("% of cluster")
    ax.set_title(f"{var} composition - {FOCUS_SPACE} / {FOCUS_WINDOW}")
    ax.legend(fontsize=6, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    save(fig, panel)
    plt.show()

### Kaplan-Meier by cluster

In [ ]:
name = "km_by_cluster"
try:
    from lifelines import KaplanMeierFitter

    surv = clinical_data.load_survival()
    labels = common.load_labels(FOCUS_SPACE, FOCUS_WINDOW)
    df = labels.join(surv, on=common.PATIENT_KEY, how="inner").drop_nulls(
        ["tt_death", "death"]).filter(pl.col("tt_death") > 0)

    if df.height < 10:
        record(name, "skipped", "fewer than 10 patients with survival data")
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        times = df.get_column("tt_death").cast(pl.Float64).to_numpy() / 30.44
        events = df.get_column("death").cast(pl.Int64).to_numpy()
        groups = df.get_column("cluster").to_numpy()
        for i, cl in enumerate(sorted(set(groups))):
            m = groups == cl
            KaplanMeierFitter().fit(
                times[m], events[m], label=f"cluster {cl} (n={int(m.sum()):,})"
            ).plot_survival_function(ax=ax, ci_show=True, color=cluster_color(i))
        ax.set_xlabel("months from first treatment")
        ax.set_ylabel("overall survival")
        ax.set_title(f"OS by cluster - {FOCUS_SPACE} / {FOCUS_WINDOW}")
        ax.set_xlim(0, 120)
        fig.tight_layout()
        save(fig, name)
        plt.show()
except ImportError as e:
    record(name, "skipped", f"lifelines unavailable ({e})")

### Silhouette vs k

In [ ]:
name = "silhouette_by_k"
scan_path = common.result_path("silhouette_scan")
if not os.path.exists(scan_path):
    record(name, "skipped", "no silhouette_scan.csv")
else:
    scan = pl.read_csv(scan_path).filter(pl.col("window") == FOCUS_WINDOW)
    if not scan.height:
        record(name, "skipped", f"no scan rows for window {FOCUS_WINDOW}")
    else:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for i, space in enumerate(sorted(scan.get_column("space").unique().to_list())):
            sub = scan.filter(pl.col("space") == space).sort("k")
            ax.plot(sub.get_column("k").to_numpy(),
                    sub.get_column("silhouette").to_numpy(),
                    marker="o", ms=4, color=cluster_color(i), label=space)
        ax.set_xlabel("k")
        ax.set_ylabel("silhouette")
        ax.set_title(f"Cluster separation vs k ({FOCUS_WINDOW})")
        ax.legend(fontsize=8)
        fig.tight_layout()
        save(fig, name)
        plt.show()

### Enrichment heatmap

Top FDR-significant **categorical** variables by cluster, as the difference between each
cluster's rate and the cohort rate, in percentage points.

Continuous variables (`level = "mean"`) are deliberately excluded: their `pct` columns carry
means in the variable's own units - years for age, site counts for `N_MET_SITES` - not
percentages. Sharing one color axis across both would be a units error, and a single
wide-range continuous variable would stretch the scale until every categorical row read as
blank. See `cluster_vs_clinical.csv` for the continuous tests.

In [ ]:
name = "enrichment_heatmap"
enr_path = common.result_path("cluster_enrichment")
res_path = common.result_path("cluster_vs_clinical")
if not (os.path.exists(enr_path) and os.path.exists(res_path)):
    record(name, "skipped", "enrichment or omnibus table missing")
else:
    enr = pl.read_csv(enr_path).filter(
        (pl.col("space") == FOCUS_SPACE) & (pl.col("window") == FOCUS_WINDOW))
    sig = pl.read_csv(res_path).filter(
        (pl.col("space") == FOCUS_SPACE) & (pl.col("window") == FOCUS_WINDOW)
        & (pl.col("significant") == True)  # noqa: E712
    ).sort("fdr").head(20)

    if not sig.height or not enr.height:
        record(name, "skipped", "nothing significant to plot")
    else:
        keep = set(sig.get_column("variable").to_list())
        # Categorical levels only -- see the note above on mixed units.
        sub = enr.filter(
            pl.col("variable").is_in(list(keep)) & (pl.col("level") != "mean")
        ).drop_nulls(["pct", "pct_overall"])
        if not sub.height:
            record(name, "skipped",
                   "no significant categorical variables (continuous excluded)")
        else:
            sub = sub.with_columns(
                (pl.col("pct") - pl.col("pct_overall")).alias("delta"),
                (pl.col("variable") + pl.lit(" = ") + pl.col("level")).alias("row_label"),
            )
            rows = sorted(sub.get_column("row_label").unique().to_list())
            cols = sorted(sub.get_column("cluster").unique().to_list())
            M = np.full((len(rows), len(cols)), np.nan)
            for r in sub.iter_rows(named=True):
                M[rows.index(r["row_label"]), cols.index(r["cluster"])] = r["delta"]

            lim = np.nanmax(np.abs(M)) or 1.0
            fig, ax = plt.subplots(figsize=(1.3 * len(cols) + 5, 0.32 * len(rows) + 2))
            im = ax.imshow(M, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
            ax.set_xticks(range(len(cols)), [f"c{c}" for c in cols])
            ax.set_yticks(range(len(rows)), rows, fontsize=7)
            ax.set_title(f"Enrichment vs cohort - {FOCUS_SPACE} / {FOCUS_WINDOW}")
            fig.colorbar(im, ax=ax, label="cluster rate - cohort rate")
            fig.tight_layout()
            save(fig, name)
            plt.show()

### Panel manifest

In [ ]:
manifest_df = pl.DataFrame(manifest) if manifest else pl.DataFrame(
    schema={"panel": pl.Utf8, "status": pl.Utf8, "reason": pl.Utf8})
common.write_result(manifest_df, "figure_manifest")
print(manifest_df)
print(f"\nFigures: {common.FIGURES_DIR}")
print(f"Tables:  {common.RESULTS_DIR}")

## Caveats

- **Exploratory.** No held-out validation, and k was chosen on the same data the clusters describe.
- **FDR is within family within space.** There is no correction across the 10 space x window runs,
  so a variable significant in exactly one space is weak evidence.
- **`alltime` is not leak-free** against survival — use `pretreatment` for anything involving `tt_death`.
- **Cancer type is the usual explanation.** Check the adjusted Cox and the scatter before reading a
  cluster as a novel phenotype.